# Baseline Models Using Pretrained Sentence Embeddings

This notebook implements **baseline classification models** using **pretrained sentence embeddings** (`all-MiniLM-L6-v2`) for Yelp reviews.

### Tasks:
1. **Sentiment classification**: Negative (1–2 stars), Neutral (3 stars), Positive (4–5 stars)  
2. **Star rating prediction**: 1–5 stars

### Models included:
- Logistic Regression
- Random Forest
- Support Vector Classifier

### Features:
- Embeddings are generated from `all-MiniLM-L6-v2`
- Stratified train/test split
- Evaluation metrics: Accuracy, Macro-F1, and per-class classification reports


In [4]:
import os
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from sklearn.metrics import accuracy_score, f1_score, classification_report
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from mord import LogisticAT

In [6]:
folder_name = 'extracted-features'

# Load embeddings and metadata
X = np.load(os.path.join(folder_name, "sentencetransformer_embeddings.npy"))
meta = pd.read_csv(os.path.join(folder_name, "sentencetransformer_meta.csv"))

sample_size = min(500000, len(meta))
data_idx = meta.sample(sample_size, random_state=42).index

X_sample = X[data_idx]
y_sent = meta.loc[data_idx, 'sentiment'].values
y_star = meta.loc[data_idx, 'stars'].values

# Train/test split
X_train, X_test, y_sent_train, y_sent_test = train_test_split(X_sample, y_sent, test_size=0.1, random_state=42, stratify=y_sent)
_, _, y_star_train, y_star_test = train_test_split(X_sample, y_star, test_size=0.1, random_state=42, stratify=y_star)


In [12]:
def train_baseline_models(X_train, y_train, X_test, y_test):    
    models = {
            'Logistic Regression': LogisticRegression(max_iter=1000),
            'SVM': SVC(kernel='linear', max_iter=2000),
            'Random Forest': RandomForestClassifier(n_estimators=200, max_features=100, max_depth=25),
            'Ordinal Logistic Regression': LogisticAT(max_iter=2000)
        }
    results = {}
    
    for name, model in models.items():
        print(f"\nTraining {name}...")
        model.fit(X_train, y_train)
        y_pred = model.predict(X_test)
        
        # Calculate metrics
        accuracy = accuracy_score(y_test, y_pred)
        macro_f1 = f1_score(y_test, y_pred, average='macro')
        
        # Per-class metrics
        report = classification_report(y_test, y_pred, output_dict=True)
        
        results[name] = {
            'model': model,
            'accuracy': accuracy,
            'macro_f1': macro_f1,
            'predictions': y_pred,
            'classification_report': report
        }
        
        print(f"Accuracy: {accuracy:.4f}")
        print(f"Macro-F1: {macro_f1:.4f}")
    
    return results


In [13]:
# Sentiment baseline
print("\n--- Sentiment Baseline ---")
sent_model_results = train_baseline_models(X_train, y_sent_train, X_test, y_sent_test)

# Star baseline
print("\n--- Star Baseline (1-5) ---")
star_model_results = train_baseline_models(X_train, y_star_train, X_test, y_star_test)


--- Sentiment Baseline ---

Training Logistic Regression...
Accuracy: 0.8287
Macro-F1: 0.6395

Training SVM...


/Users/juliasober/anaconda3/lib/python3.11/site-packages/sklearn/svm/_base.py:297: ConvergenceWarning: Solver terminated early (max_iter=2000).  Consider pre-processing your data with StandardScaler or MinMaxScaler.
  warnings.warn(


Accuracy: 0.6698
Macro-F1: 0.4146

Training Random Forest...
